In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline, Pipeline, FeatureUnion
from sklearn.preprocessing import OrdinalEncoder, FunctionTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.preprocessing import MultiLabelBinarizer
from typing import Dict, List
import unicodedata
from sklearn.svm import LinearSVC

In [6]:
# ============================================================================
# DATA LOADING AND SPLITTING
# ============================================================================

def load_and_prepare_data(filepath):
    """Load data and rename columns"""
    df = pd.read_csv(filepath)
    df.columns = [
        'id',
        'best_tasks_free',
        'acad_tasks_rating',
        'best_tasks_select',
        'subopt_freq_rating',
        'subopt_tasks_select',
        'subopt_tasks_free',
        'evidence_freq_rating',
        'verify_freq_rating',
        'verify_method_free',
        'target'
    ]
    return df


def split_data(df):
    """Split data by student ID to prevent leakage"""
    students = df['id'].unique()
    train_idx, test_idx = train_test_split(
        students, test_size=0.3, random_state=22
    )

    train_df = df[df['id'].isin(train_idx)]
    test_df = df[df['id'].isin(test_idx)]

    train_groups = train_df['id'].values
    test_groups = test_df['id'].values

    x_train = train_df.drop(columns=['target', 'id'])
    y_train = train_df['target']

    x_test = test_df.drop(columns=['target', 'id'])
    y_test = test_df['target']

    return x_train, x_test, y_train, y_test, train_groups, test_groups

In [7]:
# ============================================================================
# COLUMN DEFINITIONS
# ============================================================================

text_cols = [
    'best_tasks_free',
    'subopt_tasks_free',
    'verify_method_free',
]

ord_cols = [
    'acad_tasks_rating',
    'subopt_freq_rating',
    'evidence_freq_rating',
    'verify_freq_rating',
]

likert_categories = [
    '1 — Not at all likely',
    '2 — Unlikely',
    '3 — Neutral / Unsure',
    '4 — Likely',
    '5 — Very likely'
]

freq_categories = [
    '1 — Never',
    '2 — Rarely',
    '3 — Sometimes',
    '4 — Often',
    '5 — Very often'
]

ord_categories = [
    likert_categories,  # acad_tasks_rating
    freq_categories,    # subopt_freq_rating
    freq_categories,    # evidence_freq_rating
    freq_categories,    # verify_freq_rating
]

cat_cols = [
    'best_tasks_select',
    'subopt_tasks_select',
]

cat_multi_select_choices = [
    "Explaining complex concepts simply",
    "Drafting professional text (e.g., emails, résumés)",
    "Brainstorming or generating creative ideas",
    "Writing or editing essays/reports",
    "Converting content between formats (e.g., LaTeX)",
    "Writing or debugging code",
    "Data processing or analysis",
    "Math computations",
]


In [9]:
# ============================================================================
# TEXT PREPROCESSING - SEPARATE PIPELINES
# ============================================================================

class ColumnExtractor(BaseEstimator, TransformerMixin):
    """Extract a single column and convert to list of strings"""
    
    def __init__(self, column_index):
        self.column_index = column_index
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        # X is a DataFrame with text columns
        X = pd.DataFrame(X)
        col = X.iloc[:, self.column_index]
        # Convert to list of strings, handling NaN
        return [str(x) if pd.notna(x) else '' for x in col]


def preprocess_text_separate():
    """
    Create SEPARATE TF-IDF vectorizers for each text column
    This allows different vocabulary importance per question
    """
    return FeatureUnion([
        # best_tasks_free - most informative (what tasks?)
        ('best_tasks', Pipeline([
            ('extract', ColumnExtractor(column_index=0)),
            ('tfidf', TfidfVectorizer(
                max_features=800,
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.75,
                stop_words='english'
            ))
        ])),
        
        # subopt_tasks_free - medium informative (suboptimal responses?)
        ('subopt_tasks', Pipeline([
            ('extract', ColumnExtractor(column_index=1)),
            ('tfidf', TfidfVectorizer(
                max_features=400,
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.75,
                stop_words='english'
            ))
        ])),
        
        # verify_method_free - medium informative (how verify?)
        ('verify_method', Pipeline([
            ('extract', ColumnExtractor(column_index=2)),
            ('tfidf', TfidfVectorizer(
                max_features=400,
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.75,
                stop_words='english'
            ))
        ])),
    ])

In [10]:
# ============================================================================
# COMBINED TEXT PREPROCESSING (ORIGINAL APPROACH FOR COMPARISON)
# ============================================================================

class MakeCorpus(BaseEstimator, TransformerMixin):
    """Combine all text columns into single corpus"""
    
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        return X.agg(' '.join, axis=1).tolist()


def preprocess_text_combined():
    """Original approach - combine all text columns"""
    return Pipeline([
        ('imputer', SimpleImputer(strategy="constant", fill_value="")),
        ('combiner', MakeCorpus()),
        ('tfidf', TfidfVectorizer(
            max_features=2000,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.75,
            stop_words='english'
        ))
    ])


# ============================================================================
# ORDINAL AND CATEGORICAL PREPROCESSING
# ============================================================================

def preprocess_ord():
    """Ordinal encoding for Likert/frequency scales"""
    return make_pipeline(
        SimpleImputer(strategy="most_frequent"),
        OrdinalEncoder(categories=ord_categories)
    )


def _normalize(s: str) -> str:
    """Normalize unicode, collapse spaces, strip"""
    s = unicodedata.normalize("NFKC", str(s))
    s = " ".join(s.split())
    return s.strip()


def _split_top_level_commas(s: str) -> List[str]:
    """
    Split on commas that are NOT inside parentheses.
    Example: "Drafting ... (e.g., emails, résumés), Math computations"
    -> ["Drafting ... (e.g., emails, résumés)", "Math computations"]
    """
    parts = []
    buf = []
    depth = 0
    for ch in s:
        if ch == '(':
            depth += 1
            buf.append(ch)
        elif ch == ')':
            depth = max(0, depth - 1)
            buf.append(ch)
        elif ch == ',' and depth == 0:
            parts.append("".join(buf))
            buf = []
        else:
            buf.append(ch)
    if buf:
        parts.append("".join(buf))
    return parts


class MultiSelectBinarizerPerColumn(BaseEstimator, TransformerMixin):
    """
    One-hot encode multi-select columns using safe split
    """
    def __init__(self, classes: List[str]):
        self.classes = classes

    def _parse_cell(self, x) -> List[str]:
        if pd.isna(x) or (isinstance(x, str) and x.strip() == ""):
            return []
        norm = _normalize(x)
        pieces = [_normalize(p) for p in _split_top_level_commas(norm)]
        return [p for p in pieces if p in self.classes_]

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.classes_ = [_normalize(c) for c in self.classes]
        self.mlbs_: Dict[str, MultiLabelBinarizer] = {}
        self.col_to_outcols_: Dict[str, List[str]] = {}

        for col in X.columns:
            mlb = MultiLabelBinarizer(classes=self.classes_)
            mlb.fit([[]])
            self.mlbs_[col] = mlb
            self.col_to_outcols_[col] = [f"{col}__{c}" for c in mlb.classes_]
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        blocks = []
        for col in X.columns:
            mlb = self.mlbs_[col]
            outcols = self.col_to_outcols_[col]
            is_missing = X[col].isna()
            lists = X[col].apply(self._parse_cell)
            arr = mlb.transform(lists)
            df_bin = pd.DataFrame(arr, columns=outcols, index=X.index)
            df_bin.loc[is_missing, :] = np.nan
            blocks.append(df_bin)
        return pd.concat(blocks, axis=1)


def preprocess_cat():
    """Multi-select categorical preprocessing"""
    return make_pipeline(
        MultiSelectBinarizerPerColumn(classes=cat_multi_select_choices),
        SimpleImputer(strategy="constant", fill_value=0)
    )

In [11]:
# ============================================================================
# FULL PREPROCESSOR CREATION
# ============================================================================

def create_preprocessor(use_separate_text=True):
    """
    Create full preprocessor
    
    Args:
        use_separate_text: If True, use separate TF-IDF for each text column
                          If False, combine all text columns (original approach)
    """
    if use_separate_text:
        text_preprocessor = preprocess_text_separate()
        print("Using SEPARATE TF-IDF vectorizers for each text column")
    else:
        text_preprocessor = preprocess_text_combined()
        print("Using COMBINED TF-IDF (original approach)")
    
    transformers = [
        ("text", text_preprocessor, text_cols),
        ("ord", preprocess_ord(), ord_cols),
        ("cat", preprocess_cat(), cat_cols),
    ]
    
    return ColumnTransformer(transformers=transformers)


# ============================================================================
# MAIN TRAINING PIPELINE
# ============================================================================

def train_and_compare(df, use_separate_text=True):
    """
    Train models and compare performance
    
    Args:
        df: DataFrame with data
        use_separate_text: Whether to use separate or combined text features
    """
    print("=" * 80)
    print(f"Training with {'SEPARATE' if use_separate_text else 'COMBINED'} text features")
    print("=" * 80)
    
    # Split data
    x_train, x_test, y_train, y_test, train_groups, test_groups = split_data(df)
    print(f"\nDataset split: {len(x_train)} train, {len(x_test)} test")
    
    # Create preprocessor
    preprocessor = create_preprocessor(use_separate_text=use_separate_text)
    
    # Create pipeline with Logistic Regression
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
    ])
    
    # Grid search for Logistic Regression
    print("\n" + "=" * 80)
    print("LOGISTIC REGRESSION")
    print("=" * 80)
    
    param_grid_lr = {'classifier__C': [0.001, 0.01, 0.1, 1.0]}
    cv = GroupKFold(n_splits=5)
    
    grid_search_lr = GridSearchCV(
        pipeline, 
        param_grid_lr, 
        cv=cv, 
        scoring='accuracy',
        n_jobs=-1, 
        verbose=1
    )
    
    grid_search_lr.fit(x_train, y_train, groups=train_groups)
    
    print(f"\nBest parameters: {grid_search_lr.best_params_}")
    print(f"Best CV score: {grid_search_lr.best_score_:.4f}")
    
    best_lr = grid_search_lr.best_estimator_
    train_acc_lr = best_lr.score(x_train, y_train)
    print(f"Training accuracy: {train_acc_lr:.4f}")
    
    # Linear SVC
    print("\n" + "=" * 80)
    print("LINEAR SVC")
    print("=" * 80)
    
    best_preprocessor = best_lr.named_steps['preprocessor']
    
    pipeline_svc = Pipeline([
        ('preprocessor', best_preprocessor),
        ('classifier', LinearSVC(dual=True))
    ])
    
    param_grid_svc = {
        'classifier__C': [0.01, 0.1, 1.0, 10.0],
        'classifier__max_iter': [5000, 10000]
    }
    
    grid_search_svc = GridSearchCV(
        pipeline_svc,
        param_grid_svc,
        cv=cv,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )
    
    grid_search_svc.fit(x_train, y_train, groups=train_groups)
    
    print(f"\nBest parameters: {grid_search_svc.best_params_}")
    print(f"Best CV score: {grid_search_svc.best_score_:.4f}")
    
    best_svc = grid_search_svc.best_estimator_
    train_acc_svc = best_svc.score(x_train, y_train)
    print(f"Training accuracy: {train_acc_svc:.4f}")
    
    # Summary
    print("\n" + "=" * 80)
    print("SUMMARY")
    print("=" * 80)
    print(f"Logistic Regression - CV: {grid_search_lr.best_score_:.4f}, Train: {train_acc_lr:.4f}")
    print(f"Linear SVC          - CV: {grid_search_svc.best_score_:.4f}, Train: {train_acc_svc:.4f}")
    
    return {
        'lr_cv': grid_search_lr.best_score_,
        'lr_train': train_acc_lr,
        'svc_cv': grid_search_svc.best_score_,
        'svc_train': train_acc_svc,
        'best_lr': best_lr,
        'best_svc': best_svc
    }

In [ ]:
# SETUP

TRAINING_DATA = "/Users/vic/Desktop/csc311/project_proposal-starter-files-pergolav/training_data_clean.csv"



In [ ]:
# ============================================================================
# COMPARISON FUNCTION
# ============================================================================

def compare_approaches(df):
    """Compare separate vs combined text features"""
    
    print("\n" + "=" * 80)
    print("EXPERIMENT: SEPARATE vs COMBINED TEXT FEATURES")
    print("=" * 80)
    
    # Test with SEPARATE text features
    results_separate = train_and_compare(df, use_separate_text=True)
    
    print("\n\n")
    
    # Test with COMBINED text features (original)
    results_combined = train_and_compare(df, use_separate_text=False)
    
    # Final comparison
    print("\n" + "=" * 80)
    print("FINAL COMPARISON")
    print("=" * 80)
    
    print("\nSEPARATE Text Features:")
    print(f"  Logistic Regression CV: {results_separate['lr_cv']:.4f}")
    print(f"  Linear SVC CV:          {results_separate['svc_cv']:.4f}")
    
    print("\nCOMBINED Text Features (Original):")
    print(f"  Logistic Regression CV: {results_combined['lr_cv']:.4f}")
    print(f"  Linear SVC CV:          {results_combined['svc_cv']:.4f}")
    
    # Determine winner
    best_separate = max(results_separate['lr_cv'], results_separate['svc_cv'])
    best_combined = max(results_combined['lr_cv'], results_combined['svc_cv'])
    
    print("\n" + "=" * 80)
    if best_separate > best_combined:
        improvement = (best_separate - best_combined) * 100
        print(f"✓ SEPARATE features WIN by {improvement:.2f} percentage points!")
        print("  Recommendation: Use separate TF-IDF for each text column")
    elif best_combined > best_separate:
        improvement = (best_combined - best_separate) * 100
        print(f"✓ COMBINED features WIN by {improvement:.2f} percentage points!")
        print("  Recommendation: Stick with original combined approach")
    else:
        print("✓ TIE - both approaches perform equally")
    print("=" * 80)
    
    return results_separate, results_combined


# ============================================================================
# USAGE
# ============================================================================

if __name__ == "__main__":
    # Load data
    TRAINING_DATA = "/Users/vic/Desktop/csc311/project_proposal-starter-files-pergolav/training_data_clean.csv"
    df = load_and_prepare_data(TRAINING_DATA)
    
    print(f"Dataset loaded: {len(df)} samples, {df['target'].nunique()} classes")
    print(f"Class distribution:\n{df['target'].value_counts()}")
    
    # Just train with separate features
    # results = train_and_compare(df, use_separate_text=True)
    
    # Compare both approaches (RECOMMENDED)
    results_sep, results_comb = compare_approaches(df)

Dataset loaded: 825 samples, 3 classes
Class distribution:
target
ChatGPT    275
Claude     275
Gemini     275
Name: count, dtype: int64

EXPERIMENT: SEPARATE vs COMBINED TEXT FEATURES
Training with SEPARATE text features

Dataset split: 576 train, 249 test
Using SEPARATE TF-IDF vectorizers for each text column

LOGISTIC REGRESSION
Fitting 5 folds for each of 4 candidates, totalling 20 fits

Best parameters: {'classifier__C': 0.1}
Best CV score: 0.6769
Training accuracy: 0.7483

LINEAR SVC
Fitting 5 folds for each of 8 candidates, totalling 40 fits


/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(



Best parameters: {'classifier__C': 0.1, 'classifier__max_iter': 5000}
Best CV score: 0.6665
Training accuracy: 0.9236

SUMMARY
Logistic Regression - CV: 0.6769, Train: 0.7483
Linear SVC          - CV: 0.6665, Train: 0.9236



Training with COMBINED text features

Dataset split: 576 train, 249 test
Using COMBINED TF-IDF (original approach)

LOGISTIC REGRESSION
Fitting 5 folds for each of 4 candidates, totalling 20 fits

Best parameters: {'classifier__C': 1.0}
Best CV score: 0.6700
Training accuracy: 0.8368

LINEAR SVC
Fitting 5 folds for each of 8 candidates, totalling 40 fits


/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(



Best parameters: {'classifier__C': 0.1, 'classifier__max_iter': 5000}
Best CV score: 0.6750
Training accuracy: 0.8038

SUMMARY
Logistic Regression - CV: 0.6700, Train: 0.8368
Linear SVC          - CV: 0.6750, Train: 0.8038

FINAL COMPARISON

SEPARATE Text Features:
  Logistic Regression CV: 0.6769
  Linear SVC CV:          0.6665

COMBINED Text Features (Original):
  Logistic Regression CV: 0.6700
  Linear SVC CV:          0.6750

✓ SEPARATE features WIN by 0.19 percentage points!
  Recommendation: Use separate TF-IDF for each text column


In [ ]:
"""
Updated text handling with SEPARATE TF-IDF vectorizers for each text column
This allows the model to learn that different words matter in different contexts
"""

import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline, Pipeline, FeatureUnion
from sklearn.preprocessing import OrdinalEncoder, FunctionTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.preprocessing import MultiLabelBinarizer
from typing import Dict, List
import unicodedata
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier



# ============================================================================
# DATA LOADING AND SPLITTING
# ============================================================================

def load_and_prepare_data(filepath):
    """Load data and rename columns"""
    df = pd.read_csv(filepath)
    df.columns = [
        'id',
        'best_tasks_free',
        'acad_tasks_rating',
        'best_tasks_select',
        'subopt_freq_rating',
        'subopt_tasks_select',
        'subopt_tasks_free',
        'evidence_freq_rating',
        'verify_freq_rating',
        'verify_method_free',
        'target'
    ]
    return df


def split_data(df):
    """Split data by student ID to prevent leakage"""
    students = df['id'].unique()
    train_idx, test_idx = train_test_split(
        students, test_size=0.3, random_state=22
    )

    train_df = df[df['id'].isin(train_idx)]
    test_df = df[df['id'].isin(test_idx)]

    train_groups = train_df['id'].values
    test_groups = test_df['id'].values

    x_train = train_df.drop(columns=['target', 'id'])
    y_train = train_df['target']

    x_test = test_df.drop(columns=['target', 'id'])
    y_test = test_df['target']

    return x_train, x_test, y_train, y_test, train_groups, test_groups


# ============================================================================
# COLUMN DEFINITIONS
# ============================================================================

text_cols = [
    'best_tasks_free',
    'subopt_tasks_free',
    'verify_method_free',
]

ord_cols = [
    'acad_tasks_rating',
    'subopt_freq_rating',
    'evidence_freq_rating',
    'verify_freq_rating',
]

likert_categories = [
    '1 — Not at all likely',
    '2 — Unlikely',
    '3 — Neutral / Unsure',
    '4 — Likely',
    '5 — Very likely'
]

freq_categories = [
    '1 — Never',
    '2 — Rarely',
    '3 — Sometimes',
    '4 — Often',
    '5 — Very often'
]

ord_categories = [
    likert_categories,  # acad_tasks_rating
    freq_categories,    # subopt_freq_rating
    freq_categories,    # evidence_freq_rating
    freq_categories,    # verify_freq_rating
]

cat_cols = [
    'best_tasks_select',
    'subopt_tasks_select',
]

cat_multi_select_choices = [
    "Explaining complex concepts simply",
    "Drafting professional text (e.g., emails, résumés)",
    "Brainstorming or generating creative ideas",
    "Writing or editing essays/reports",
    "Converting content between formats (e.g., LaTeX)",
    "Writing or debugging code",
    "Data processing or analysis",
    "Math computations",
]


# ============================================================================
# TEXT PREPROCESSING - SEPARATE PIPELINES
# ============================================================================

class ColumnExtractor(BaseEstimator, TransformerMixin):
    """Extract a single column and convert to list of strings"""
    
    def __init__(self, column_index):
        self.column_index = column_index
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        # X is a DataFrame with text columns
        X = pd.DataFrame(X)
        col = X.iloc[:, self.column_index]
        # Convert to list of strings, handling NaN
        return [str(x) if pd.notna(x) else '' for x in col]


def preprocess_text_separate():
    """
    Create SEPARATE TF-IDF vectorizers for each text column
    This allows different vocabulary importance per question
    """
    return FeatureUnion([
        # best_tasks_free - most informative (what tasks?)
        ('best_tasks', Pipeline([
            ('extract', ColumnExtractor(column_index=0)),
            ('tfidf', TfidfVectorizer(
                max_features=800,
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.75,
                stop_words='english'
            ))
        ])),
        
        # subopt_tasks_free - medium informative (suboptimal responses?)
        ('subopt_tasks', Pipeline([
            ('extract', ColumnExtractor(column_index=1)),
            ('tfidf', TfidfVectorizer(
                max_features=400,
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.75,
                stop_words='english'
            ))
        ])),
        
        # verify_method_free - medium informative (how verify?)
        ('verify_method', Pipeline([
            ('extract', ColumnExtractor(column_index=2)),
            ('tfidf', TfidfVectorizer(
                max_features=400,
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.75,
                stop_words='english'
            ))
        ])),
    ])


# ============================================================================
# COMBINED TEXT PREPROCESSING (ORIGINAL APPROACH FOR COMPARISON)
# ============================================================================

class MakeCorpus(BaseEstimator, TransformerMixin):
    """Combine all text columns into single corpus"""
    
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        return X.agg(' '.join, axis=1).tolist()


def preprocess_text_combined():
    """Original approach - combine all text columns"""
    return Pipeline([
        ('imputer', SimpleImputer(strategy="constant", fill_value="")),
        ('combiner', MakeCorpus()),
        ('tfidf', TfidfVectorizer(
            max_features=2000,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.75,
            stop_words='english'
        ))
    ])


# ============================================================================
# ORDINAL AND CATEGORICAL PREPROCESSING
# ============================================================================

def preprocess_ord():
    """Ordinal encoding for Likert/frequency scales"""
    return make_pipeline(
        SimpleImputer(strategy="most_frequent"),
        OrdinalEncoder(categories=ord_categories)
    )


def _normalize(s: str) -> str:
    """Normalize unicode, collapse spaces, strip"""
    s = unicodedata.normalize("NFKC", str(s))
    s = " ".join(s.split())
    return s.strip()


def _split_top_level_commas(s: str) -> List[str]:
    """
    Split on commas that are NOT inside parentheses.
    Example: "Drafting ... (e.g., emails, résumés), Math computations"
    -> ["Drafting ... (e.g., emails, résumés)", "Math computations"]
    """
    parts = []
    buf = []
    depth = 0
    for ch in s:
        if ch == '(':
            depth += 1
            buf.append(ch)
        elif ch == ')':
            depth = max(0, depth - 1)
            buf.append(ch)
        elif ch == ',' and depth == 0:
            parts.append("".join(buf))
            buf = []
        else:
            buf.append(ch)
    if buf:
        parts.append("".join(buf))
    return parts


class MultiSelectBinarizerPerColumn(BaseEstimator, TransformerMixin):
    """
    One-hot encode multi-select columns using safe split
    """
    def __init__(self, classes: List[str]):
        self.classes = classes

    def _parse_cell(self, x) -> List[str]:
        if pd.isna(x) or (isinstance(x, str) and x.strip() == ""):
            return []
        norm = _normalize(x)
        pieces = [_normalize(p) for p in _split_top_level_commas(norm)]
        return [p for p in pieces if p in self.classes_]

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.classes_ = [_normalize(c) for c in self.classes]
        self.mlbs_: Dict[str, MultiLabelBinarizer] = {}
        self.col_to_outcols_: Dict[str, List[str]] = {}

        for col in X.columns:
            mlb = MultiLabelBinarizer(classes=self.classes_)
            mlb.fit([[]])
            self.mlbs_[col] = mlb
            self.col_to_outcols_[col] = [f"{col}__{c}" for c in mlb.classes_]
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        blocks = []
        for col in X.columns:
            mlb = self.mlbs_[col]
            outcols = self.col_to_outcols_[col]
            is_missing = X[col].isna()
            lists = X[col].apply(self._parse_cell)
            arr = mlb.transform(lists)
            df_bin = pd.DataFrame(arr, columns=outcols, index=X.index)
            df_bin.loc[is_missing, :] = np.nan
            blocks.append(df_bin)
        return pd.concat(blocks, axis=1)


def preprocess_cat():
    """Multi-select categorical preprocessing"""
    return make_pipeline(
        MultiSelectBinarizerPerColumn(classes=cat_multi_select_choices),
        SimpleImputer(strategy="constant", fill_value=0)
    )


# ============================================================================
# FULL PREPROCESSOR CREATION
# ============================================================================

def create_preprocessor(use_separate_text=True):
    """
    Create full preprocessor
    
    Args:
        use_separate_text: If True, use separate TF-IDF for each text column
                          If False, combine all text columns (original approach)
    """
    if use_separate_text:
        text_preprocessor = preprocess_text_separate()
        print("Using SEPARATE TF-IDF vectorizers for each text column")
    else:
        text_preprocessor = preprocess_text_combined()
        print("Using COMBINED TF-IDF (original approach)")
    
    transformers = [
        ("text", text_preprocessor, text_cols),
        ("ord", preprocess_ord(), ord_cols),
        ("cat", preprocess_cat(), cat_cols),
    ]
    
    return ColumnTransformer(transformers=transformers)


# ============================================================================
# MAIN TRAINING PIPELINE
# ============================================================================

def train_and_compare(df, use_separate_text=True):
    """
    Train models and compare performance
    
    Args:
        df: DataFrame with data
        use_separate_text: Whether to use separate or combined text features
    """
    print("=" * 80)
    print(f"Training with {'SEPARATE' if use_separate_text else 'COMBINED'} text features")
    print("=" * 80)
    
    # Split data
    x_train, x_test, y_train, y_test, train_groups, test_groups = split_data(df)
    print(f"\nDataset split: {len(x_train)} train, {len(x_test)} test")
    
    # Create preprocessor
    preprocessor = create_preprocessor(use_separate_text=use_separate_text)
    
    # Create pipeline with Logistic Regression
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
    ])
    
    # Grid search for Logistic Regression
    print("\n" + "=" * 80)
    print("LOGISTIC REGRESSION")
    print("=" * 80)
    
    param_grid_lr = {'classifier__C': [0.001, 0.01, 0.1, 1.0]}
    cv = GroupKFold(n_splits=5)
    
    grid_search_lr = GridSearchCV(
        pipeline, 
        param_grid_lr, 
        cv=cv, 
        scoring='accuracy',
        n_jobs=-1, 
        verbose=1
    )
    
    grid_search_lr.fit(x_train, y_train, groups=train_groups)
    
    print(f"\nBest parameters: {grid_search_lr.best_params_}")
    print(f"Best CV score: {grid_search_lr.best_score_:.4f}")
    
    best_lr = grid_search_lr.best_estimator_
    train_acc_lr = best_lr.score(x_train, y_train)
    print(f"Training accuracy: {train_acc_lr:.4f}")
    
    # Linear SVC
    print("\n" + "=" * 80)
    print("LINEAR SVC")
    print("=" * 80)
    
    best_preprocessor = best_lr.named_steps['preprocessor']
    
    pipeline_svc = Pipeline([
        ('preprocessor', best_preprocessor),
        ('classifier', LinearSVC(dual=True))
    ])
    
    param_grid_svc = {
        'classifier__C': [0.01, 0.1, 1.0, 10.0],
        'classifier__max_iter': [5000, 10000]
    }
    
    grid_search_svc = GridSearchCV(
        pipeline_svc,
        param_grid_svc,
        cv=cv,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )
    
    grid_search_svc.fit(x_train, y_train, groups=train_groups)
    
    print(f"\nBest parameters: {grid_search_svc.best_params_}")
    print(f"Best CV score: {grid_search_svc.best_score_:.4f}")
    
    best_svc = grid_search_svc.best_estimator_
    train_acc_svc = best_svc.score(x_train, y_train)
    print(f"Training accuracy: {train_acc_svc:.4f}")


    print("\n" + "=" * 80)
    print("RANDOM FOREST")
    print("=" * 80)

    pipeline_rf = Pipeline([
        ('preprocessor', best_preprocessor),
        ('classifier', RandomForestClassifier())
    ])

    param_grid_rf = {
        'classifier__n_estimators': [300, 500],
        'classifier__max_depth': [10, 20, None],
        'classifier__min_samples_split': [2, 5]
    }

    grid_search_rf = GridSearchCV(
        pipeline_rf,
        param_grid_rf,
        cv=cv,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )

    grid_search_rf.fit(x_train, y_train, groups=train_groups)

    best_rf = grid_search_rf.best_estimator_
    train_acc_rf = best_rf.score(x_train, y_train)

    print(f"Best RF params: {grid_search_rf.best_params_}")
    print(f"Best RF CV: {grid_search_rf.best_score_:.4f}")
    print(f"RF train accuracy: {train_acc_rf:.4f}")

    
    # Summary
    print("\n" + "=" * 80)
    print("SUMMARY")
    print("=" * 80)
    print(f"Logistic Regression - CV: {grid_search_lr.best_score_:.4f}, Train: {train_acc_lr:.4f}")
    print(f"Linear SVC          - CV: {grid_search_svc.best_score_:.4f}, Train: {train_acc_svc:.4f}")
    
    return {
        'lr_cv': grid_search_lr.best_score_,
        'lr_train': train_acc_lr,
        'svc_cv': grid_search_svc.best_score_,
        'svc_train': train_acc_svc,
        'best_lr': best_lr,
        'best_svc': best_svc,
        'rf_cv': grid_search_rf.best_score_,
        'rf_train': train_acc_rf
    }


# ============================================================================
# COMPARISON FUNCTION
# ============================================================================

def compare_approaches(df):
    """Compare separate vs combined text features"""
    
    print("\n" + "=" * 80)
    print("EXPERIMENT: SEPARATE vs COMBINED TEXT FEATURES")
    print("=" * 80)
    
    # Test with SEPARATE text features
    results_separate = train_and_compare(df, use_separate_text=True)
    
    print("\n\n")
    
    # Test with COMBINED text features (original)
    results_combined = train_and_compare(df, use_separate_text=False)
    
    # Final comparison
    print("\n" + "=" * 80)
    print("FINAL COMPARISON")
    print("=" * 80)
    
    print("\nSEPARATE Text Features:")
    print(f"  Logistic Regression CV: {results_separate['lr_cv']:.4f}")
    print(f"  Linear SVC CV:          {results_separate['svc_cv']:.4f}")
    print(f"  Random Forest CV:       {results_separate['rf_cv']:.4f}")
    
    print("\nCOMBINED Text Features (Original):")
    print(f"  Logistic Regression CV: {results_combined['lr_cv']:.4f}")
    print(f"  Linear SVC CV:          {results_combined['svc_cv']:.4f}")
    print(f"  Random Forest CV:       {results_combined['rf_cv']:.4f}")
    
    # Determine winner
    best_separate = max(results_separate['lr_cv'], results_separate['svc_cv'], results_separate['rf_cv'])
    best_combined = max(results_combined['lr_cv'], results_combined['svc_cv'], results_combined['rf_cv'])
    
    print("\n" + "=" * 80)
    print("RECOMMENDATION")
    print("=" * 80)
    
    if best_separate > best_combined:
        improvement = (best_separate - best_combined) * 100
        print(f"✓ SEPARATE features WIN by {improvement:.2f} percentage points!")
        print("  Recommendation: Use separate TF-IDF for each text column")
        
        # Find which model is best with separate
        if results_separate['rf_cv'] == best_separate:
            print(f"  Best model: Random Forest (CV: {results_separate['rf_cv']:.4f})")
        elif results_separate['svc_cv'] == best_separate:
            print(f"  Best model: Linear SVC (CV: {results_separate['svc_cv']:.4f})")
        else:
            print(f"  Best model: Logistic Regression (CV: {results_separate['lr_cv']:.4f})")
            
    elif best_combined > best_separate:
        improvement = (best_combined - best_separate) * 100
        print(f"✓ COMBINED features WIN by {improvement:.2f} percentage points!")
        print("  Recommendation: Stick with original combined approach")
        
        # Find which model is best with combined
        if results_combined['rf_cv'] == best_combined:
            print(f"  Best model: Random Forest (CV: {results_combined['rf_cv']:.4f})")
        elif results_combined['svc_cv'] == best_combined:
            print(f"  Best model: Linear SVC (CV: {results_combined['svc_cv']:.4f})")
        else:
            print(f"  Best model: Logistic Regression (CV: {results_combined['lr_cv']:.4f})")
    else:
        print("✓ TIE - both approaches perform equally")
    
    print("=" * 80)
    
    # Check overfitting for Random Forest
    print("\nRandom Forest Overfitting Analysis:")
    print(f"  SEPARATE: Train={results_separate['rf_train']:.4f}, CV={results_separate['rf_cv']:.4f}, Gap={((results_separate['rf_train']-results_separate['rf_cv'])*100):.2f}%")
    print(f"  COMBINED: Train={results_combined['rf_train']:.4f}, CV={results_combined['rf_cv']:.4f}, Gap={((results_combined['rf_train']-results_combined['rf_cv'])*100):.2f}%")
    
    return results_separate, results_combined


# ============================================================================
# USAGE
# ============================================================================

if __name__ == "__main__":
    # Load data
    TRAINING_DATA = "/Users/vic/Desktop/csc311/project_proposal-starter-files-pergolav/training_data_clean.csv"
    df = load_and_prepare_data(TRAINING_DATA)
    
    print(f"Dataset loaded: {len(df)} samples, {df['target'].nunique()} classes")
    print(f"Class distribution:\n{df['target'].value_counts()}")
    
    # Just train with separate features
    # results = train_and_compare(df, use_separate_text=True)
    
    # Compare both approaches (RECOMMENDED)
    results_sep, results_comb = compare_approaches(df)

Dataset loaded: 825 samples, 3 classes
Class distribution:
target
ChatGPT    275
Claude     275
Gemini     275
Name: count, dtype: int64

EXPERIMENT: SEPARATE vs COMBINED TEXT FEATURES
Training with SEPARATE text features

Dataset split: 576 train, 249 test
Using SEPARATE TF-IDF vectorizers for each text column

LOGISTIC REGRESSION
Fitting 5 folds for each of 4 candidates, totalling 20 fits

Best parameters: {'classifier__C': 0.1}
Best CV score: 0.6769
Training accuracy: 0.7483

LINEAR SVC
Fitting 5 folds for each of 8 candidates, totalling 40 fits


/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(



Best parameters: {'classifier__C': 0.1, 'classifier__max_iter': 5000}
Best CV score: 0.6665
Training accuracy: 0.9236

RANDOM FOREST
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best RF params: {'classifier__max_depth': None, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 300}
Best RF CV: 0.6771
RF train accuracy: 0.9948

SUMMARY
Logistic Regression - CV: 0.6769, Train: 0.7483
Linear SVC          - CV: 0.6665, Train: 0.9236



Training with COMBINED text features

Dataset split: 576 train, 249 test
Using COMBINED TF-IDF (original approach)

LOGISTIC REGRESSION
Fitting 5 folds for each of 4 candidates, totalling 20 fits

Best parameters: {'classifier__C': 1.0}
Best CV score: 0.6700
Training accuracy: 0.8368

LINEAR SVC
Fitting 5 folds for each of 8 candidates, totalling 40 fits


/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(



Best parameters: {'classifier__C': 0.1, 'classifier__max_iter': 5000}
Best CV score: 0.6750
Training accuracy: 0.8038

RANDOM FOREST
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best RF params: {'classifier__max_depth': 20, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 500}
Best RF CV: 0.6754
RF train accuracy: 0.9913

SUMMARY
Logistic Regression - CV: 0.6700, Train: 0.8368
Linear SVC          - CV: 0.6750, Train: 0.8038

FINAL COMPARISON

SEPARATE Text Features:
  Logistic Regression CV: 0.6769
  Linear SVC CV:          0.6665
  Random Forest CV:       0.6771

COMBINED Text Features (Original):
  Logistic Regression CV: 0.6700
  Linear SVC CV:          0.6750
  Random Forest CV:       0.6754

RECOMMENDATION
✓ SEPARATE features WIN by 0.16 percentage points!
  Recommendation: Use separate TF-IDF for each text column
  Best model: Random Forest (CV: 0.6771)

Random Forest Overfitting Analysis:
  SEPARATE: Train=0.9948, CV=0.6771, Gap=31.77%
  COMBINED:

In [ ]:
"""
STACKING INTEGRATION
Text model + Tabular model → Meta-classifier
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# ---------------------------------------------------------------
# USE EXISTING PREPROCESSORS
# ---------------------------------------------------------------


# ======================================================================
# STACKING IMPLEMENTATION
# ======================================================================

def train_stacked_model(df, text_cols, ord_cols, cat_cols):
    """
    Stack:
    - Base Model A: Text → LinearSVC
    - Base Model B: Tabular → RandomForest
    - Meta-model: Logistic Regression
    
    WITH:
      ✔ CV accuracy for text model
      ✔ CV accuracy for tabular model
      ✔ CV accuracy for stacked model
      ✔ Final Test Accuracy
    """

    # -----------------------------------------
    # Split by student ID to prevent leakage
    # -----------------------------------------
    students = df['id'].unique()
    train_ids, test_ids = train_test_split(students, test_size=0.3, random_state=22)

    train_df = df[df['id'].isin(train_ids)]
    test_df  = df[df['id'].isin(test_ids)]

    X_train = train_df.drop(columns=['target', 'id'])
    y_train = train_df['target']
    X_test  = test_df.drop(columns=['target', 'id'])
    y_test  = test_df['target']

    groups = train_df['id'].values
    classes = sorted(y_train.unique())

    # ==========================================================
    # Preprocessors
    # ==========================================================
    text_pre = ColumnTransformer([
        ('text', preprocess_text_separate(), text_cols)
    ])

    tab_pre = ColumnTransformer([
        ('ord', preprocess_ord(), ord_cols),
        ('cat', preprocess_cat(), cat_cols)
    ])

    text_model = Pipeline([
        ('pre', text_pre),
        ('clf', LinearSVC(C=0.1))
    ])

    tab_model = Pipeline([
        ('pre', tab_pre),
        ('clf', RandomForestClassifier(
            n_estimators=400,
            max_depth=20,
            min_samples_split=5,
            random_state=42
        ))
    ])

    # ==========================================================
    # BASE MODEL CV ACCURACY
    # ==========================================================
    K = 5
    gkf = GroupKFold(n_splits=K)

    cv_scores_text = []
    cv_scores_tab  = []

    for tr, val in gkf.split(X_train, y_train, groups=groups):
        X_tr, X_val = X_train.iloc[tr], X_train.iloc[val]
        y_tr, y_val = y_train.iloc[tr], y_train.iloc[val]

        # --- Text model ---
        text_model.fit(X_tr[text_cols], y_tr)
        pred_text = text_model.predict(X_val[text_cols])
        cv_scores_text.append(accuracy_score(y_val, pred_text))

        # --- Tabular model ---
        tab_model.fit(X_tr[ord_cols + cat_cols], y_tr)
        pred_tab = tab_model.predict(X_val[ord_cols + cat_cols])
        cv_scores_tab.append(accuracy_score(y_val, pred_tab))

    print("\n====================================")
    print(" BASE MODEL CROSS-VALIDATION RESULTS")
    print("====================================")
    print(f"Text Model CV Acc:  mean={np.mean(cv_scores_text):.4f}, std={np.std(cv_scores_text):.4f}")
    print(f"Tab  Model CV Acc:  mean={np.mean(cv_scores_tab):.4f}, std={np.std(cv_scores_tab):.4f}")

    # ==========================================================
    # STACKED MODEL: OOF PREDICTIONS
    # ==========================================================
    oof_text = np.zeros((len(X_train), len(classes)))
    oof_tab  = np.zeros((len(X_train), len(classes)))

    # To compute CV for meta-model:
    stack_cv_scores = []

    for tr, val in gkf.split(X_train, y_train, groups=groups):

        X_tr, X_val = X_train.iloc[tr], X_train.iloc[val]
        y_tr, y_val = y_train.iloc[tr], y_train.iloc[val]

        # Train text model
        text_model.fit(X_tr[text_cols], y_tr)
        s_text = text_model.decision_function(X_val[text_cols])
        s_text = (s_text - s_text.min(axis=1, keepdims=True))
        s_text = s_text / s_text.sum(axis=1, keepdims=True)
        oof_text[val] = s_text

        # Train tab model
        tab_model.fit(X_tr[ord_cols + cat_cols], y_tr)
        s_tab = tab_model.predict_proba(X_val[ord_cols + cat_cols])
        oof_tab[val] = s_tab

        # Prepare meta inputs for this fold
        tab_val_proc = tab_pre.fit_transform(X_tr).shape
        tab_val_proc = tab_pre.transform(X_val)

        X_meta_fold = np.hstack([s_text, s_tab, tab_val_proc])

        # Train & evaluate meta-model on this fold
        meta_clf = LogisticRegression(max_iter=2000)
        meta_clf.fit(
            np.hstack([oof_text[tr], oof_tab[tr], tab_pre.fit_transform(X_train.iloc[tr][ord_cols + cat_cols])]),
            y_tr
        )

        stack_preds = meta_clf.predict(X_meta_fold)
        stack_cv_scores.append(accuracy_score(y_val, stack_preds))

    print("\n====================================")
    print(" STACKED MODEL CROSS-VALIDATION")
    print("====================================")
    print(f"Stacked CV Acc:      mean={np.mean(stack_cv_scores):.4f}, std={np.std(stack_cv_scores):.4f}")

    # ==========================================================
    # TRAIN FINAL META MODEL ON ALL TRAINING DATA
    # ==========================================================
    tab_train_proc = tab_pre.fit_transform(X_train)

    X_meta_train = np.hstack([oof_text, oof_tab, tab_train_proc])

    meta_final = LogisticRegression(max_iter=2000)
    meta_final.fit(X_meta_train, y_train)

    # ==========================================================
    # TEST SET EVALUATION
    # ==========================================================
    st = text_model.decision_function(X_test[text_cols])
    st = (st - st.min(axis=1, keepdims=True))
    st = st / st.sum(axis=1, keepdims=True)

    tab_test_proc = tab_pre.transform(X_test)
    tab_probs_test = tab_model.predict_proba(X_test[ord_cols + cat_cols])

    X_meta_test = np.hstack([st, tab_probs_test, tab_test_proc])

    final_preds = meta_final.predict(X_meta_test)
    test_acc = accuracy_score(y_test, final_preds)

    print("\n====================================")
    print(" FINAL TEST PERFORMANCE")
    print("====================================")
    print(f"Final Test Accuracy: {test_acc:.4f}")

    return {
        "text_cv": cv_scores_text,
        "tab_cv": cv_scores_tab,
        "stack_cv_scores": stack_cv_scores,
        "test_accuracy": test_acc,
        "meta_model": meta_final
    }




# ======================================================================
# MAIN ENTRY POINT
# ======================================================================

if __name__ == "__main__":

    TRAINING_DATA = "/Users/vic/Desktop/csc311/project_proposal-starter-files-pergolav/training_data_clean.csv"

    df = pd.read_csv(TRAINING_DATA)
    df.columns = [
        'id',
        'best_tasks_free',
        'acad_tasks_rating',
        'best_tasks_select',
        'subopt_freq_rating',
        'subopt_tasks_select',
        'subopt_tasks_free',
        'evidence_freq_rating',
        'verify_freq_rating',
        'verify_method_free',
        'target'
    ]

    print(f"Loaded {len(df)} entries.")
    print(df['target'].value_counts())

    # Define columns
    text_cols = [
        'best_tasks_free',
        'subopt_tasks_free',
        'verify_method_free'
    ]

    ord_cols = [
        'acad_tasks_rating',
        'subopt_freq_rating',
        'evidence_freq_rating',
        'verify_freq_rating'
    ]

    cat_cols = [
        'best_tasks_select',
        'subopt_tasks_select'
    ]

    # Train stacking model
    results = train_stacked_model(df, text_cols, ord_cols, cat_cols)


Loaded 825 entries.
target
ChatGPT    275
Claude     275
Gemini     275
Name: count, dtype: int64

 BASE MODEL CROSS-VALIDATION RESULTS
Text Model CV Acc:  mean=0.6076, std=0.0084
Tab  Model CV Acc:  mean=0.6368, std=0.0449

 STACKED MODEL CROSS-VALIDATION
Stacked CV Acc:      mean=0.6613, std=0.0308

 FINAL TEST PERFORMANCE
Final Test Accuracy: 0.6867


In [ ]:
"""
Integrated tuning + stacking
Replace or append to your existing script. Assumes your preprocessors
(preprocess_text_separate, preprocess_ord, preprocess_cat, etc.) are defined.
"""

import os
import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import FunctionTransformer

# -------------------------
# CONFIG
# -------------------------
TRAINING_DATA = "/Users/vic/Desktop/csc311/project_proposal-starter-files-pergolav/training_data_clean.csv"
RESULTS_DIR = "model_tuning_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
RANDOM_STATE = 22
N_JOBS = -1

# -------------------------
# TUNERS
# -------------------------

def tune_text_model(df, text_cols, groups, n_splits=5):
    """
    Tune TF-IDF + LinearSVC using GroupKFold.
    Returns best_params (grid.best_params_) and the fitted GridSearchCV object.
    """
    # Collapse multiple text columns into a single column for the vectorizer input
    X_text = df[text_cols].apply(lambda r: " ".join([str(x) for x in r.values]), axis=1)
    y = df['target']

    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(lowercase=True, stop_words='english')),
        ('svc', LinearSVC(dual=True, random_state=RANDOM_STATE))
    ])

    param_grid = {
        'tfidf__max_features': [5000, 10000],            # reduce grid if slow
        'tfidf__ngram_range': [(1,1), (1,2)],
        'tfidf__min_df': [1, 2],
        'tfidf__sublinear_tf': [True, False],
        'tfidf__use_idf': [True],

        'svc__C': [0.01, 0.1, 1, 3],
        'svc__loss': ['squared_hinge'],  # squared_hinge generally converges better for multi-class with OvR
        'svc__max_iter': [5000]
    }

    gkf = GroupKFold(n_splits=n_splits)
    grid = GridSearchCV(pipe, param_grid, cv=gkf.split(X_text, y, groups), scoring='accuracy',
                        n_jobs=N_JOBS, verbose=2, refit=True)
    print("Tuning TEXT model (this can take a while)...")
    grid.fit(X_text, y)
    print("Best text params:", grid.best_params_)
    return grid.best_params_, grid

def tune_tabular_model(df, ord_cols, cat_cols, groups, n_splits=5):
    """
    Tune RandomForest for tabular features using GroupKFold.
    Returns best_params and fitted GridSearchCV.
    """
    X_tab = df[ord_cols + cat_cols]
    y = df['target']

    # Preprocessor for tabular-only model (reuses your helper function creators)
    tab_pre = ColumnTransformer([
        ('ord', preprocess_ord(), ord_cols),
        ('cat', preprocess_cat(), cat_cols)
    ])

    pipe = Pipeline([
        ('pre', tab_pre),
        ('rf', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1))  # GridSearchCV uses jobs
    ])

    param_grid = {
        'rf__n_estimators': [300, 600],
        'rf__max_depth': [10, 20, None],
        'rf__min_samples_split': [2, 5],
        'rf__min_samples_leaf': [1, 2],
        'rf__max_features': ['sqrt']
    }

    gkf = GroupKFold(n_splits=n_splits)
    grid = GridSearchCV(pipe, param_grid, cv=gkf.split(X_tab, y, groups), scoring='accuracy',
                        n_jobs=N_JOBS, verbose=2, refit=True)
    print("Tuning TABULAR model (RandomForest)...")
    grid.fit(X_tab, y)
    print("Best tabular params:", grid.best_params_)
    return grid.best_params_, grid

def tune_meta_model(X_meta, y_meta):
    """
    Optional small tuner for meta model. X_meta and y_meta are numpy arrays.
    """
    pipe = Pipeline([('clf', LogisticRegression(max_iter=2000))])
    param_grid = {
        'clf__C': [0.01, 0.1, 1, 3]
    }
    grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=N_JOBS, verbose=1, refit=True)
    grid.fit(X_meta, y_meta)
    print("Best meta params:", grid.best_params_)
    return grid.best_params_, grid

# -------------------------
# STACKING + EVAL
# -------------------------
def run_tune_and_stack(df, text_cols, ord_cols, cat_cols,
                       do_tune_text=True, do_tune_tab=True, do_tune_meta=True):
    """
    Orchestrates tuning and stacking. Returns results dict and saves models/params.
    """

    # split students to create train/test (no leakage)
    students = df['id'].unique()
    train_ids, test_ids = train_test_split(students, test_size=0.3, random_state=RANDOM_STATE)
    train_df = df[df['id'].isin(train_ids)].reset_index(drop=True)
    test_df  = df[df['id'].isin(test_ids)].reset_index(drop=True)

    X_train = train_df.drop(columns=['target', 'id'])
    y_train = train_df['target']
    groups_train = train_df['id'].values

    X_test = test_df.drop(columns=['target', 'id'])
    y_test = test_df['target']

    classes = sorted(y_train.unique())
    n_classes = len(classes)

    # ----------------------------
    # Tune Text model
    # ----------------------------
    if do_tune_text:
        best_text_params, grid_text = tune_text_model(train_df, text_cols, groups_train)
        # Extract fitted vectorizer + classifier pipeline for usage
        best_text_pipe = grid_text.best_estimator_
    else:
        # default text pipeline (fallback)
        default_tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2), min_df=1, sublinear_tf=True)
        best_text_pipe = Pipeline([('tfidf', default_tfidf), ('svc', LinearSVC(C=0.1, max_iter=5000, random_state=RANDOM_STATE))])
        best_text_pipe.fit(train_df[text_cols].apply(lambda r: " ".join([str(x) for x in r.values]), axis=1), y_train)

    # ----------------------------
    # Tune Tabular model
    # ----------------------------
    if do_tune_tab:
        best_tab_params, grid_tab = tune_tabular_model(train_df, ord_cols, cat_cols, groups_train)
        best_tab_pipe = grid_tab.best_estimator_
    else:
        tab_pre = ColumnTransformer([
            ('ord', preprocess_ord(), ord_cols),
            ('cat', preprocess_cat(), cat_cols)
        ])
        best_tab_pipe = Pipeline([
            ('pre', tab_pre),
            ('rf', RandomForestClassifier(n_estimators=400, max_depth=20, random_state=RANDOM_STATE))
        ])
        best_tab_pipe.fit(train_df[ord_cols + cat_cols], y_train)

    # ----------------------------
    # Evaluate base-model CV (informative)
    # ----------------------------
    K = 5
    gkf = GroupKFold(n_splits=K)
    text_cv_scores = []
    tab_cv_scores = []

    X_text_full = train_df[text_cols].apply(lambda r: " ".join([str(x) for x in r.values]), axis=1)
    X_tab_full = train_df[ord_cols + cat_cols]

    for tr_idx, val_idx in gkf.split(train_df, y_train, groups_train):
        # TEXT
        best_text_pipe.fit(X_text_full.iloc[tr_idx], y_train.iloc[tr_idx])
        pred_text = best_text_pipe.predict(X_text_full.iloc[val_idx])
        text_cv_scores.append(accuracy_score(y_train.iloc[val_idx], pred_text))

        # TAB
        best_tab_pipe.fit(X_tab_full.iloc[tr_idx], y_train.iloc[tr_idx])
        pred_tab = best_tab_pipe.predict(X_tab_full.iloc[val_idx])
        tab_cv_scores.append(accuracy_score(y_train.iloc[val_idx], pred_tab))

    print("\nBase model CV (text): mean={:.4f} std={:.4f}".format(np.mean(text_cv_scores), np.std(text_cv_scores)))
    print("Base model CV (tab):  mean={:.4f} std={:.4f}".format(np.mean(tab_cv_scores), np.std(tab_cv_scores)))

    # ----------------------------
    # Build OOF predictions for stacking
    # ----------------------------
    oof_text = np.zeros((len(train_df), n_classes))
    oof_tab  = np.zeros((len(train_df), n_classes))

    tab_pre_full = ColumnTransformer([
        ('ord', preprocess_ord(), ord_cols),
        ('cat', preprocess_cat(), cat_cols)
    ])

    for tr_idx, val_idx in gkf.split(train_df, y_train, groups_train):
        # Train base models on fold
        # TEXT
        best_text_pipe.fit(X_text_full.iloc[tr_idx], y_train.iloc[tr_idx])
        scores_text = best_text_pipe.decision_function(X_text_full.iloc[val_idx])
        # normalize to pseudo-probs (row-wise)
        scores_text = (scores_text - scores_text.min(axis=1, keepdims=True))
        scores_text = scores_text / (scores_text.sum(axis=1, keepdims=True) + 1e-12)
        oof_text[val_idx] = scores_text

        # TAB
        best_tab_pipe.fit(X_tab_full.iloc[tr_idx], y_train.iloc[tr_idx])
        probs_tab = best_tab_pipe.predict_proba(X_tab_full.iloc[val_idx])
        oof_tab[val_idx] = probs_tab

    # ----------------------------
    # Prepare meta inputs & optionally tune meta model
    # ----------------------------
    # tabular processed features for entire train set
    tab_train_proc = tab_pre_full.fit_transform(train_df)
    X_meta = np.hstack([oof_text, oof_tab, tab_train_proc])

    if do_tune_meta:
        best_meta_params, grid_meta = tune_meta_model(X_meta, y_train)
        meta_clf = grid_meta.best_estimator_
    else:
        meta_clf = LogisticRegression(max_iter=3000)

    meta_clf.fit(X_meta, y_train)

    # ----------------------------
    # Evaluate stacked model on test set
    # ----------------------------
    # build final components on full training set first
    # fit text pipeline on whole training set
    best_text_pipe.fit(train_df[text_cols].apply(lambda r: " ".join([str(x) for x in r.values]), axis=1), y_train)
    best_tab_pipe.fit(train_df[ord_cols + cat_cols], y_train)
    tab_test_proc = tab_pre_full.transform(test_df)

    # test-time text scores
    X_test_text = test_df[text_cols].apply(lambda r: " ".join([str(x) for x in r.values]), axis=1)
    st = best_text_pipe.decision_function(X_test_text)
    st = (st - st.min(axis=1, keepdims=True))
    st = st / (st.sum(axis=1, keepdims=True) + 1e-12)

    # test-time tab probs
    tab_probs_test = best_tab_pipe.predict_proba(test_df[ord_cols + cat_cols])

    X_meta_test = np.hstack([st, tab_probs_test, tab_test_proc])
    final_preds = meta_clf.predict(X_meta_test)
    test_acc = accuracy_score(test_df['target'], final_preds)

    # ----------------------------
    # Save results + artifacts
    # ----------------------------
    results = {
        "text_cv_scores": [float(s) for s in text_cv_scores],
        "tab_cv_scores": [float(s) for s in tab_cv_scores],
        "test_accuracy": float(test_acc),

        # ONLY save JSON-safe parameter dicts
        "best_text_params": (best_text_params if do_tune_text else None),
        "best_tab_params": (best_tab_params if do_tune_tab else None),
        "best_meta_params": (best_meta_params if do_tune_meta else None)

    }


    with open(os.path.join(RESULTS_DIR, "stacking_results.json"), "w") as f:
        json.dump(results, f, indent=2)

    # save fitted pipelines using joblib if you like (optional)
    try:
        import joblib
        joblib.dump(best_text_pipe, os.path.join(RESULTS_DIR, "best_text_pipe.joblib"))
        joblib.dump(best_tab_pipe, os.path.join(RESULTS_DIR, "best_tab_pipe.joblib"))
        joblib.dump(meta_clf, os.path.join(RESULTS_DIR, "meta_clf.joblib"))
    except Exception as e:
        print("joblib not available or save failed:", e)

    print("\n=== FINAL TEST ACCURACY (STACKED): {:.4f} ===".format(test_acc))
    return results

# -------------------------
# MAIN
# -------------------------
if __name__ == "__main__":
    # Load data (uses same header as before)
    df = pd.read_csv(TRAINING_DATA)
    df.columns = [
        'id',
        'best_tasks_free',
        'acad_tasks_rating',
        'best_tasks_select',
        'subopt_freq_rating',
        'subopt_tasks_select',
        'subopt_tasks_free',
        'evidence_freq_rating',
        'verify_freq_rating',
        'verify_method_free',
        'target'
    ]

    text_cols = [
        'best_tasks_free',
        'subopt_tasks_free',
        'verify_method_free'
    ]
    ord_cols = [
        'acad_tasks_rating',
        'subopt_freq_rating',
        'evidence_freq_rating',
        'verify_freq_rating'
    ]
    cat_cols = [
        'best_tasks_select',
        'subopt_tasks_select'
    ]

    # Run full tune + stacking (may take time)
    results = run_tune_and_stack(df, text_cols, ord_cols, cat_cols,
                                 do_tune_text=True, do_tune_tab=True, do_tune_meta=True)

    print("Saved results to:", RESULTS_DIR)


Tuning TEXT model (this can take a while)...
Fitting 5 folds for each of 64 candidates, totalling 320 fits
[CV] END svc__C=0.01, svc__loss=squared_hinge, svc__max_iter=5000, tfidf__max_features=5000, tfidf__min_df=1, tfidf__ngram_range=(1, 1), tfidf__sublinear_tf=True, tfidf__use_idf=True; total time=   0.0s
[CV] END svc__C=0.01, svc__loss=squared_hinge, svc__max_iter=5000, tfidf__max_features=5000, tfidf__min_df=1, tfidf__ngram_range=(1, 1), tfidf__sublinear_tf=True, tfidf__use_idf=True; total time=   0.0s
[CV] END svc__C=0.01, svc__loss=squared_hinge, svc__max_iter=5000, tfidf__max_features=5000, tfidf__min_df=1, tfidf__ngram_range=(1, 1), tfidf__sublinear_tf=True, tfidf__use_idf=True; total time=   0.0s
[CV] END svc__C=0.01, svc__loss=squared_hinge, svc__max_iter=5000, tfidf__max_features=5000, tfidf__min_df=1, tfidf__ngram_range=(1, 1), tfidf__sublinear_tf=True, tfidf__use_idf=True; total time=   0.0s
[CV] END svc__C=0.01, svc__loss=squared_hinge, svc__max_iter=5000, tfidf__max_fea

In [41]:
"""
Enhanced classification script with improved feature engineering and model architecture
"""

import os
import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin
import re

# -------------------------
# CONFIG
# -------------------------
TRAINING_DATA = "/Users/vic/Desktop/csc311/project_proposal-starter-files-pergolav/training_data_clean.csv"
RESULTS_DIR = "model_tuning_results_enhanced"
os.makedirs(RESULTS_DIR, exist_ok=True)
RANDOM_STATE = 22
N_JOBS = -1

# -------------------------
# ENHANCED FEATURE ENGINEERING
# -------------------------

class TextFeatureExtractor(BaseEstimator, TransformerMixin):
    """Extract additional features from text responses"""
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        # Handle pandas Series or array
        if hasattr(X, 'values'):
            X = X.values
        
        features = []
        for text in X:
            text = str(text).lower()
            feat = {
                'length': len(text),
                'word_count': len(text.split()),
                'avg_word_len': np.mean([len(w) for w in text.split()]) if text.split() else 0,
                'sentence_count': len(re.split(r'[.!?]+', text)),
                
                # Model-specific keywords
                'mentions_model': int(bool(re.search(r'\[this model\]|\[another model\]', text))),
                'has_comparison': int(bool(re.search(r'better|worse|compared|versus|vs', text))),
                'negative_sentiment': int(bool(re.search(r'bad|poor|terrible|wrong|incorrect|fail', text))),
                'positive_sentiment': int(bool(re.search(r'good|great|excellent|perfect|love', text))),
                
                # Technical indicators
                'mentions_code': int(bool(re.search(r'\bcode\b|\bcoding\b|\bdebug', text))),
                'mentions_math': int(bool(re.search(r'\bmath\b|\bcalculation\b|\bcomputation', text))),
                'mentions_writing': int(bool(re.search(r'\bwrit\w+|\bessay|\bprose', text))),
                
                # Response characteristics
                'mentions_verbose': int(bool(re.search(r'verbose|wordy|lengthy|too much', text))),
                'mentions_simple': int(bool(re.search(r'simple|dumb|trivial|obvious', text))),
                'mentions_wrong': int(bool(re.search(r'wrong|error|mistake|incorrect|hallucin', text))),
            }
            features.append(list(feat.values()))
        
        return np.array(features)

class SurveyPatternFeatures(BaseEstimator, TransformerMixin):
    """Extract patterns from survey response combinations"""
    
    def __init__(self, ord_cols, cat_cols):
        self.ord_cols = ord_cols
        self.cat_cols = cat_cols
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        # X is a DataFrame with survey columns
        features = []
        
        for idx, row in X.iterrows():
            # Get ordinal values with defaults
            ratings = []
            for col in self.ord_cols:
                if col in row:
                    val = extract_numeric(row[col])
                    if pd.notna(val):
                        ratings.append(val)
            
            feat = {
                # Rating consistency
                'rating_variance': np.var(ratings) if len(ratings) > 1 else 0,
                'rating_mean': np.mean(ratings) if ratings else 3,
                
                # Engagement level (based on selections)
                'best_task_count': 0,
                'subopt_task_count': 0,
            }
            
            # Count selected tasks
            if 'best_tasks_select' in row:
                best_tasks = str(row['best_tasks_select']) if pd.notna(row['best_tasks_select']) else ''
                feat['best_task_count'] = len([x for x in best_tasks.split(',') if x.strip()])
            
            if 'subopt_tasks_select' in row:
                subopt_tasks = str(row['subopt_tasks_select']) if pd.notna(row['subopt_tasks_select']) else ''
                feat['subopt_task_count'] = len([x for x in subopt_tasks.split(',') if x.strip()])
            
            # User satisfaction indicators
            acad_rating = extract_numeric(row.get('acad_tasks_rating', 3))
            acad_rating = acad_rating if pd.notna(acad_rating) else 3
            
            subopt_rating = extract_numeric(row.get('subopt_freq_rating', 3))
            subopt_rating = subopt_rating if pd.notna(subopt_rating) else 3
            
            verify_rating = extract_numeric(row.get('verify_freq_rating', 3))
            verify_rating = verify_rating if pd.notna(verify_rating) else 3
            
            feat['high_likelihood'] = int(acad_rating >= 4)
            feat['frequent_subopt'] = int(subopt_rating >= 4)
            feat['frequent_verify'] = int(verify_rating >= 4)
            feat['critical_user'] = int(subopt_rating >= 4 and verify_rating >= 4)
            feat['satisfied_user'] = int(acad_rating >= 4 and subopt_rating <= 2)
            
            features.append(list(feat.values()))
        
        return np.array(features)

def preprocess_ord():
    """Ordinal features preprocessor - extract numeric values from text"""
    def impute_and_scale(X):
        X_copy = X.copy()
        
        # Extract numeric values from strings like "5 — Very likely"
        for col in X_copy.columns:
            X_copy[col] = X_copy[col].apply(lambda x: extract_numeric(x))
        
        # Fill missing with neutral value 3
        X_copy = X_copy.fillna(3)
        
        return StandardScaler().fit_transform(X_copy)
    
    return FunctionTransformer(impute_and_scale)

def extract_numeric(val):
    """Extract numeric value from ordinal response"""
    if pd.isna(val):
        return np.nan
    
    # If already numeric, return it
    if isinstance(val, (int, float)):
        return float(val)
    
    # Extract first number from string like "5 — Very likely"
    val_str = str(val)
    match = re.search(r'(\d+)', val_str)
    if match:
        return float(match.group(1))
    
    return np.nan

def preprocess_cat():
    """Categorical features preprocessor with better handling"""
    from sklearn.preprocessing import MultiLabelBinarizer
    
    def process_cat(X):
        result = []
        for idx, row in X.iterrows():
            row_features = []
            for col in X.columns:
                val = str(row[col]) if pd.notna(row[col]) else ''
                # Split by comma and create binary features
                items = [item.strip() for item in val.split(',') if item.strip()]
                # Create feature for each unique task type
                task_types = [
                    'Math', 'Writing', 'Data', 'Explain', 'Convert', 
                    'Draft', 'Brainstorm', 'Debug', 'code'
                ]
                for task in task_types:
                    row_features.append(int(any(task.lower() in item.lower() for item in items)))
            result.append(row_features)
        return np.array(result)
    
    return FunctionTransformer(process_cat)

# -------------------------
# IMPROVED TUNING FUNCTIONS
# -------------------------

def create_text_feature_extractor(col_name):
    """Helper to create column-specific text extractor"""
    def extract_column(X):
        return X[col_name].fillna('').astype(str).values
    return FunctionTransformer(extract_column, validate=False)

class TextColumnSelector(BaseEstimator, TransformerMixin):
    """Select and clean a text column"""
    def __init__(self, col_name):
        self.col_name = col_name
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        # Return cleaned text as list of strings
        return X[self.col_name].fillna('').astype(str).tolist()

def tune_enhanced_text_model(df, text_cols, groups, n_splits=5):
    """
    Enhanced text model with separate vectorizers per text field
    """
    y = df['target']
    
    # Create separate TF-IDF pipeline for each text column
    text_transformers = []
    for col in text_cols:
        text_transformers.append(
            (f'tfidf_{col}', 
             Pipeline([
                 ('select', TextColumnSelector(col)),
                 ('tfidf', TfidfVectorizer(
                     lowercase=True, 
                     stop_words='english',
                     max_features=3000,
                     ngram_range=(1, 2),
                     min_df=1,
                     sublinear_tf=True
                 ))
             ]),
             text_cols)  # Pass all columns, selector will pick the right one
        )
    
    # Add text feature extractor for each column
    for col in text_cols:
        text_transformers.append(
            (f'textfeat_{col}',
             TextFeatureExtractor(),
             col)
        )
    
    preprocessor = ColumnTransformer(
        text_transformers,
        remainder='drop'
    )
    
    pipe = Pipeline([
        ('features', preprocessor),
        ('clf', LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
    ])
    
    param_grid = {
        'clf__C': [0.01, 0.1, 0.5, 1.0, 3.0],
        'clf__penalty': ['l2'],
        'clf__solver': ['saga'],
        'clf__class_weight': [None, 'balanced']
    }
    
    gkf = GroupKFold(n_splits=n_splits)
    grid = GridSearchCV(
        pipe, param_grid, 
        cv=gkf.split(df[text_cols], y, groups), 
        scoring='accuracy',
        n_jobs=N_JOBS, 
        verbose=2, 
        refit=True
    )
    
    print("Tuning ENHANCED TEXT model...")
    grid.fit(df[text_cols], y)
    print("Best enhanced text params:", grid.best_params_)
    print("Best CV score:", grid.best_score_)
    
    return grid.best_params_, grid

def tune_enhanced_tabular_model(df, ord_cols, cat_cols, groups, n_splits=5):
    """
    Enhanced tabular model with gradient boosting and better features
    """
    y = df['target']
    
    # Create comprehensive preprocessor
    preprocessor = ColumnTransformer([
        ('ord', preprocess_ord(), ord_cols),
        ('cat', preprocess_cat(), cat_cols),
        ('survey_patterns', SurveyPatternFeatures(ord_cols, cat_cols), ord_cols + cat_cols)
    ])
    
    pipe = Pipeline([
        ('pre', preprocessor),
        ('clf', GradientBoostingClassifier(random_state=RANDOM_STATE))
    ])
    
    param_grid = {
        'clf__n_estimators': [200, 400],
        'clf__learning_rate': [0.05, 0.1],
        'clf__max_depth': [4, 6, 8],
        'clf__min_samples_split': [5, 10],
        'clf__min_samples_leaf': [2, 4],
        'clf__subsample': [0.8, 1.0],
        'clf__max_features': ['sqrt', 0.5]
    }
    
    gkf = GroupKFold(n_splits=n_splits)
    grid = GridSearchCV(
        pipe, param_grid,
        cv=gkf.split(df[ord_cols + cat_cols], y, groups),
        scoring='accuracy',
        n_jobs=N_JOBS,
        verbose=2,
        refit=True
    )
    
    print("Tuning ENHANCED TABULAR model...")
    grid.fit(df[ord_cols + cat_cols], y)
    print("Best enhanced tabular params:", grid.best_params_)
    print("Best CV score:", grid.best_score_)
    
    return grid.best_params_, grid

def tune_meta_model_enhanced(X_meta, y_meta):
    """Enhanced meta model with more options"""
    
    param_grid = {
        'clf__C': [0.01, 0.1, 0.5, 1, 3, 10],
        'clf__penalty': ['l2'],
        'clf__solver': ['saga'],
        'clf__class_weight': [None, 'balanced']
    }
    
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
    ])
    
    grid = GridSearchCV(
        pipe, param_grid, 
        cv=5, 
        scoring='accuracy',
        n_jobs=N_JOBS, 
        verbose=1, 
        refit=True
    )
    
    grid.fit(X_meta, y_meta)
    print("Best meta params:", grid.best_params_)
    print("Best meta CV score:", grid.best_score_)
    
    return grid.best_params_, grid

# -------------------------
# MAIN ORCHESTRATION
# -------------------------

def run_enhanced_pipeline(df, text_cols, ord_cols, cat_cols):
    """Run the complete enhanced pipeline"""
    
    # Split by students
    students = df['id'].unique()
    train_ids, test_ids = train_test_split(
        students, test_size=0.3, random_state=RANDOM_STATE
    )
    
    train_df = df[df['id'].isin(train_ids)].reset_index(drop=True)
    test_df = df[df['id'].isin(test_ids)].reset_index(drop=True)
    
    y_train = train_df['target']
    y_test = test_df['target']
    groups_train = train_df['id'].values
    
    classes = sorted(y_train.unique())
    n_classes = len(classes)
    
    print(f"\nTrain size: {len(train_df)}, Test size: {len(test_df)}")
    print(f"Classes: {classes}")
    print(f"Train distribution:\n{y_train.value_counts()}")
    
    # Tune models
    print("\n" + "="*60)
    print("TUNING TEXT MODEL")
    print("="*60)
    best_text_params, grid_text = tune_enhanced_text_model(
        train_df, text_cols, groups_train
    )
    
    print("\n" + "="*60)
    print("TUNING TABULAR MODEL")
    print("="*60)
    best_tab_params, grid_tab = tune_enhanced_tabular_model(
        train_df, ord_cols, cat_cols, groups_train
    )
    
    # Generate OOF predictions for stacking
    print("\n" + "="*60)
    print("GENERATING OOF PREDICTIONS")
    print("="*60)
    
    K = 5
    gkf = GroupKFold(n_splits=K)
    oof_text = np.zeros((len(train_df), n_classes))
    oof_tab = np.zeros((len(train_df), n_classes))
    
    text_cv_scores = []
    tab_cv_scores = []
    
    for fold, (tr_idx, val_idx) in enumerate(gkf.split(train_df, y_train, groups_train)):
        print(f"\nProcessing fold {fold+1}/{K}")
        
        # TEXT
        train_text_data = train_df.iloc[tr_idx][text_cols]
        val_text_data = train_df.iloc[val_idx][text_cols]
        
        grid_text.best_estimator_.fit(train_text_data, y_train.iloc[tr_idx])
        pred_text = grid_text.best_estimator_.predict(val_text_data)
        probs_text = grid_text.best_estimator_.predict_proba(val_text_data)
        oof_text[val_idx] = probs_text
        text_cv_scores.append(accuracy_score(y_train.iloc[val_idx], pred_text))
        
        # TAB
        tab_data_tr = train_df.iloc[tr_idx][ord_cols + cat_cols]
        tab_data_val = train_df.iloc[val_idx][ord_cols + cat_cols]
        grid_tab.best_estimator_.fit(tab_data_tr, y_train.iloc[tr_idx])
        pred_tab = grid_tab.best_estimator_.predict(tab_data_val)
        probs_tab = grid_tab.best_estimator_.predict_proba(tab_data_val)
        oof_tab[val_idx] = probs_tab
        tab_cv_scores.append(accuracy_score(y_train.iloc[val_idx], pred_tab))
    
    print(f"\nText CV: {np.mean(text_cv_scores):.4f} ± {np.std(text_cv_scores):.4f}")
    print(f"Tab CV:  {np.mean(tab_cv_scores):.4f} ± {np.std(tab_cv_scores):.4f}")
    
    # Train meta model
    print("\n" + "="*60)
    print("TRAINING META MODEL")
    print("="*60)
    
    X_meta = np.hstack([oof_text, oof_tab])
    best_meta_params, grid_meta = tune_meta_model_enhanced(X_meta, y_train)
    
    # Final test evaluation
    print("\n" + "="*60)
    print("FINAL TEST EVALUATION")
    print("="*60)
    
    # Retrain on full training set
    grid_text.best_estimator_.fit(train_df[text_cols], y_train)
    grid_tab.best_estimator_.fit(train_df[ord_cols + cat_cols], y_train)
    
    # Generate test predictions
    test_text_probs = grid_text.best_estimator_.predict_proba(test_df[text_cols])
    test_tab_probs = grid_tab.best_estimator_.predict_proba(test_df[ord_cols + cat_cols])
    
    X_meta_test = np.hstack([test_text_probs, test_tab_probs])
    final_preds = grid_meta.best_estimator_.predict(X_meta_test)
    
    test_acc = accuracy_score(y_test, final_preds)
    
    print(f"\nFinal Test Accuracy: {test_acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, final_preds, target_names=classes))
    
    # Save results
    results = {
        "text_cv_scores": [float(s) for s in text_cv_scores],
        "text_cv_mean": float(np.mean(text_cv_scores)),
        "text_cv_std": float(np.std(text_cv_scores)),
        
        "tab_cv_scores": [float(s) for s in tab_cv_scores],
        "tab_cv_mean": float(np.mean(tab_cv_scores)),
        "tab_cv_std": float(np.std(tab_cv_scores)),
        
        "test_accuracy": float(test_acc),
        
        "best_text_params": best_text_params,
        "best_tab_params": best_tab_params,
        "best_meta_params": best_meta_params
    }
    
    with open(os.path.join(RESULTS_DIR, "enhanced_results.json"), "w") as f:
        json.dump(results, f, indent=2)
    
    # Save models
    try:
        import joblib
        joblib.dump(grid_text.best_estimator_, 
                   os.path.join(RESULTS_DIR, "text_model.joblib"))
        joblib.dump(grid_tab.best_estimator_,
                   os.path.join(RESULTS_DIR, "tab_model.joblib"))
        joblib.dump(grid_meta.best_estimator_,
                   os.path.join(RESULTS_DIR, "meta_model.joblib"))
        print(f"\nModels saved to {RESULTS_DIR}")
    except Exception as e:
        print(f"Error saving models: {e}")
    
    return results

# -------------------------
# MAIN
# -------------------------

if __name__ == "__main__":
    df = pd.read_csv(TRAINING_DATA)
    df.columns = [
        'id',
        'best_tasks_free',
        'acad_tasks_rating',
        'best_tasks_select',
        'subopt_freq_rating',
        'subopt_tasks_select',
        'subopt_tasks_free',
        'evidence_freq_rating',
        'verify_freq_rating',
        'verify_method_free',
        'target'
    ]
    
    text_cols = [
        'best_tasks_free',
        'subopt_tasks_free',
        'verify_method_free'
    ]
    
    ord_cols = [
        'acad_tasks_rating',
        'subopt_freq_rating',
        'evidence_freq_rating',
        'verify_freq_rating'
    ]
    
    cat_cols = [
        'best_tasks_select',
        'subopt_tasks_select'
    ]
    
    results = run_enhanced_pipeline(df, text_cols, ord_cols, cat_cols)
    print(f"\n{'='*60}")
    print("PIPELINE COMPLETE")
    print(f"{'='*60}")


Train size: 576, Test size: 249
Classes: ['ChatGPT', 'Claude', 'Gemini']
Train distribution:
target
ChatGPT    192
Claude     192
Gemini     192
Name: count, dtype: int64

TUNING TEXT MODEL
Tuning ENHANCED TEXT model...
Fitting 5 folds for each of 10 candidates, totalling 50 fits


/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/User

[CV] END clf__C=0.01, clf__class_weight=None, clf__penalty=l2, clf__solver=saga; total time=   5.4s
[CV] END clf__C=0.01, clf__class_weight=balanced, clf__penalty=l2, clf__solver=saga; total time=   5.4s
[CV] END clf__C=0.01, clf__class_weight=None, clf__penalty=l2, clf__solver=saga; total time=   5.5s
[CV] END clf__C=0.01, clf__class_weight=None, clf__penalty=l2, clf__solver=saga; total time=   5.5s
[CV] END clf__C=0.01, clf__class_weight=balanced, clf__penalty=l2, clf__solver=saga; total time=   5.5s
[CV] END clf__C=0.01, clf__class_weight=balanced, clf__penalty=l2, clf__solver=saga; total time=   5.6s


/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV] END clf__C=0.01, clf__class_weight=None, clf__penalty=l2, clf__solver=saga; total time=   5.7s
[CV] END clf__C=0.01, clf__class_weight=None, clf__penalty=l2, clf__solver=saga; total time=   5.7s


...


Processing fold 5/5


/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(



Text CV: 0.4928 ± 0.0341
Tab CV:  0.6505 ± 0.0620

TRAINING META MODEL
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best meta params: {'clf__C': 0.01, 'clf__class_weight': None, 'clf__penalty': 'l2', 'clf__solver': 'saga'}
Best meta CV score: 0.6441079460269865

FINAL TEST EVALUATION


/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(



Final Test Accuracy: 0.6988

Classification Report:
              precision    recall  f1-score   support

     ChatGPT       0.74      0.83      0.78        83
      Claude       0.71      0.61      0.66        83
      Gemini       0.64      0.65      0.65        83

    accuracy                           0.70       249
   macro avg       0.70      0.70      0.70       249
weighted avg       0.70      0.70      0.70       249

Error saving models: Can't pickle <function preprocess_ord.<locals>.impute_and_scale at 0x16b7cac20>: it's not found as __main__.preprocess_ord.<locals>.impute_and_scale

PIPELINE COMPLETE


In [42]:
"""
Simplified but effective classification script focusing on proven improvements
"""

import os
import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin
import re

# -------------------------
# CONFIG
# -------------------------
TRAINING_DATA = "/Users/vic/Desktop/csc311/project_proposal-starter-files-pergolav/training_data_clean.csv"
RESULTS_DIR = "model_tuning_results_v2"
os.makedirs(RESULTS_DIR, exist_ok=True)
RANDOM_STATE = 22
N_JOBS = -1

# -------------------------
# HELPER FUNCTIONS
# -------------------------

def extract_numeric(val):
    """Extract numeric value from ordinal response"""
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, float)):
        return float(val)
    val_str = str(val)
    match = re.search(r'(\d+)', val_str)
    if match:
        return float(match.group(1))
    return np.nan

def preprocess_ord():
    """Ordinal features preprocessor"""
    def impute_and_scale(X):
        X_copy = X.copy()
        for col in X_copy.columns:
            X_copy[col] = X_copy[col].apply(lambda x: extract_numeric(x))
        X_copy = X_copy.fillna(3)
        return StandardScaler().fit_transform(X_copy)
    return FunctionTransformer(impute_and_scale)

def preprocess_cat():
    """Better categorical preprocessor"""
    def process_cat(X):
        result = []
        task_types = [
            'math', 'writing', 'data', 'explain', 'convert', 
            'draft', 'brainstorm', 'debug', 'code', 'editing'
        ]
        
        for idx, row in X.iterrows():
            row_features = []
            for col in X.columns:
                val = str(row[col]).lower() if pd.notna(row[col]) else ''
                # Count each task type
                for task in task_types:
                    row_features.append(int(task in val))
            result.append(row_features)
        return np.array(result)
    
    return FunctionTransformer(process_cat)

# -------------------------
# COMBINED TEXT APPROACH (SIMPLER)
# -------------------------

def tune_combined_text_model(df, text_cols, groups, n_splits=5):
    """
    Single vectorizer on combined text (your original approach but with better params)
    """
    y = df['target']
    
    # Combine text columns
    X_text = df[text_cols].fillna('').astype(str).agg(' '.join, axis=1)
    
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(
            lowercase=True, 
            stop_words='english',
            strip_accents='unicode'
        )),
        ('clf', LogisticRegression(
            max_iter=3000, 
            random_state=RANDOM_STATE,
            multi_class='multinomial'
        ))
    ])
    
    # Expanded parameter grid
    param_grid = {
        'tfidf__max_features': [3000, 5000, 8000],
        'tfidf__ngram_range': [(1,1), (1,2), (1,3)],
        'tfidf__min_df': [1, 2],
        'tfidf__max_df': [0.95, 1.0],
        'tfidf__sublinear_tf': [True, False],
        
        'clf__C': [0.05, 0.1, 0.5, 1.0, 2.0],
        'clf__penalty': ['l2'],
        'clf__solver': ['lbfgs'],
        'clf__class_weight': [None, 'balanced']
    }
    
    gkf = GroupKFold(n_splits=n_splits)
    grid = GridSearchCV(
        pipe, param_grid,
        cv=gkf.split(X_text, y, groups),
        scoring='accuracy',
        n_jobs=N_JOBS,
        verbose=2,
        refit=True
    )
    
    print("Tuning TEXT model with expanded grid...")
    grid.fit(X_text, y)
    print(f"Best text params: {grid.best_params_}")
    print(f"Best CV score: {grid.best_score_:.4f}")
    
    return grid.best_params_, grid

# -------------------------
# ENHANCED TABULAR MODEL
# -------------------------

class EnhancedTabularFeatures(BaseEstimator, TransformerMixin):
    """Add engineered features from tabular data"""
    
    def __init__(self, ord_cols):
        self.ord_cols = ord_cols
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        features = []
        
        for idx, row in X.iterrows():
            # Extract numeric ratings
            ratings = []
            for col in self.ord_cols:
                if col in X.columns:
                    val = extract_numeric(row[col])
                    if pd.notna(val):
                        ratings.append(val)
            
            if not ratings:
                ratings = [3, 3, 3, 3]
            
            feat = [
                np.mean(ratings),
                np.std(ratings),
                np.min(ratings),
                np.max(ratings),
                int(np.mean(ratings) >= 4),  # high satisfaction
                int(np.mean(ratings) <= 2),  # low satisfaction
            ]
            
            # Count selected tasks
            for col in ['best_tasks_select', 'subopt_tasks_select']:
                if col in X.columns:
                    val = str(row[col]) if pd.notna(row[col]) else ''
                    task_count = len([x for x in val.split(',') if x.strip()])
                    feat.append(task_count)
                else:
                    feat.append(0)
            
            features.append(feat)
        
        return np.array(features)

def tune_enhanced_tabular(df, ord_cols, cat_cols, groups, n_splits=5):
    """Enhanced tabular with GradientBoosting"""
    y = df['target']
    X_tab = df[ord_cols + cat_cols]
    
    preprocessor = ColumnTransformer([
        ('ord', preprocess_ord(), ord_cols),
        ('cat', preprocess_cat(), cat_cols),
        ('enhanced', EnhancedTabularFeatures(ord_cols), ord_cols + cat_cols)
    ])
    
    pipe = Pipeline([
        ('pre', preprocessor),
        ('clf', GradientBoostingClassifier(random_state=RANDOM_STATE))
    ])
    
    param_grid = {
        'clf__n_estimators': [200, 300, 500],
        'clf__learning_rate': [0.05, 0.1, 0.15],
        'clf__max_depth': [3, 4, 5],
        'clf__min_samples_split': [5, 10, 20],
        'clf__min_samples_leaf': [2, 4],
        'clf__subsample': [0.8, 0.9, 1.0],
        'clf__max_features': ['sqrt', 0.5]
    }
    
    gkf = GroupKFold(n_splits=n_splits)
    grid = GridSearchCV(
        pipe, param_grid,
        cv=gkf.split(X_tab, y, groups),
        scoring='accuracy',
        n_jobs=N_JOBS,
        verbose=2,
        refit=True
    )
    
    print("Tuning TABULAR model...")
    grid.fit(X_tab, y)
    print(f"Best tabular params: {grid.best_params_}")
    print(f"Best CV score: {grid.best_score_:.4f}")
    
    return grid.best_params_, grid

# -------------------------
# STACKING WITH VOTING ENSEMBLE
# -------------------------

def run_improved_pipeline(df, text_cols, ord_cols, cat_cols):
    """Run improved pipeline with better architecture"""
    
    # Split by students
    students = df['id'].unique()
    train_ids, test_ids = train_test_split(
        students, test_size=0.3, random_state=RANDOM_STATE
    )
    
    train_df = df[df['id'].isin(train_ids)].reset_index(drop=True)
    test_df = df[df['id'].isin(test_ids)].reset_index(drop=True)
    
    y_train = train_df['target']
    y_test = test_df['target']
    groups_train = train_df['id'].values
    
    classes = sorted(y_train.unique())
    n_classes = len(classes)
    
    print(f"\nTrain: {len(train_df)} samples, Test: {len(test_df)} samples")
    print(f"Classes: {classes}")
    print(f"Train distribution:\n{y_train.value_counts()}\n")
    
    # =============================
    # TUNE MODELS
    # =============================
    
    print("="*60)
    print("PHASE 1: TUNING BASE MODELS")
    print("="*60)
    
    best_text_params, grid_text = tune_combined_text_model(
        train_df, text_cols, groups_train
    )
    
    print("\n" + "="*60)
    best_tab_params, grid_tab = tune_enhanced_tabular(
        train_df, ord_cols, cat_cols, groups_train
    )
    
    # =============================
    # EVALUATE BASE MODELS
    # =============================
    
    print("\n" + "="*60)
    print("PHASE 2: BASE MODEL CV PERFORMANCE")
    print("="*60)
    
    K = 5
    gkf = GroupKFold(n_splits=K)
    
    text_cv_scores = []
    tab_cv_scores = []
    
    X_text_full = train_df[text_cols].fillna('').astype(str).agg(' '.join, axis=1)
    X_tab_full = train_df[ord_cols + cat_cols]
    
    for fold, (tr_idx, val_idx) in enumerate(gkf.split(train_df, y_train, groups_train)):
        # Text model
        grid_text.best_estimator_.fit(X_text_full.iloc[tr_idx], y_train.iloc[tr_idx])
        pred = grid_text.best_estimator_.predict(X_text_full.iloc[val_idx])
        text_cv_scores.append(accuracy_score(y_train.iloc[val_idx], pred))
        
        # Tabular model
        grid_tab.best_estimator_.fit(X_tab_full.iloc[tr_idx], y_train.iloc[tr_idx])
        pred = grid_tab.best_estimator_.predict(X_tab_full.iloc[val_idx])
        tab_cv_scores.append(accuracy_score(y_train.iloc[val_idx], pred))
    
    print(f"Text CV:    {np.mean(text_cv_scores):.4f} ± {np.std(text_cv_scores):.4f}")
    print(f"Tabular CV: {np.mean(tab_cv_scores):.4f} ± {np.std(tab_cv_scores):.4f}")
    
    # =============================
    # STACKING ENSEMBLE
    # =============================
    
    print("\n" + "="*60)
    print("PHASE 3: STACKING ENSEMBLE")
    print("="*60)
    
    # Generate OOF predictions for meta-model
    oof_text = np.zeros((len(train_df), n_classes))
    oof_tab = np.zeros((len(train_df), n_classes))
    
    for tr_idx, val_idx in gkf.split(train_df, y_train, groups_train):
        grid_text.best_estimator_.fit(X_text_full.iloc[tr_idx], y_train.iloc[tr_idx])
        oof_text[val_idx] = grid_text.best_estimator_.predict_proba(X_text_full.iloc[val_idx])
        
        grid_tab.best_estimator_.fit(X_tab_full.iloc[tr_idx], y_train.iloc[tr_idx])
        oof_tab[val_idx] = grid_tab.best_estimator_.predict_proba(X_tab_full.iloc[val_idx])
    
    # Train meta-model
    X_meta = np.hstack([oof_text, oof_tab])
    
    # Try different meta-models
    meta_models = {
        'logistic': LogisticRegression(max_iter=3000, C=1.0, random_state=RANDOM_STATE),
        'logistic_balanced': LogisticRegression(max_iter=3000, C=1.0, class_weight='balanced', random_state=RANDOM_STATE),
        'gradient_boost': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=RANDOM_STATE)
    }
    
    best_meta_name = None
    best_meta_score = 0
    best_meta_model = None
    
    for name, model in meta_models.items():
        scores = []
        for tr_idx, val_idx in gkf.split(X_meta, y_train, groups_train):
            model.fit(X_meta[tr_idx], y_train.iloc[tr_idx])
            pred = model.predict(X_meta[val_idx])
            scores.append(accuracy_score(y_train.iloc[val_idx], pred))
        
        mean_score = np.mean(scores)
        print(f"Meta-model '{name}': {mean_score:.4f} ± {np.std(scores):.4f}")
        
        if mean_score > best_meta_score:
            best_meta_score = mean_score
            best_meta_name = name
            best_meta_model = model
    
    print(f"\nBest meta-model: {best_meta_name}")
    
    # Train final meta-model on all training data
    best_meta_model.fit(X_meta, y_train)
    
    # =============================
    # FINAL TEST EVALUATION
    # =============================
    
    print("\n" + "="*60)
    print("PHASE 4: FINAL TEST EVALUATION")
    print("="*60)
    
    # Retrain base models on full training set
    grid_text.best_estimator_.fit(X_text_full, y_train)
    grid_tab.best_estimator_.fit(X_tab_full, y_train)
    
    # Test predictions
    X_test_text = test_df[text_cols].fillna('').astype(str).agg(' '.join, axis=1)
    X_test_tab = test_df[ord_cols + cat_cols]
    
    test_text_probs = grid_text.best_estimator_.predict_proba(X_test_text)
    test_tab_probs = grid_tab.best_estimator_.predict_proba(X_test_tab)
    
    # Individual model predictions
    pred_text_only = grid_text.best_estimator_.predict(X_test_text)
    pred_tab_only = grid_tab.best_estimator_.predict(X_test_tab)
    
    acc_text_only = accuracy_score(y_test, pred_text_only)
    acc_tab_only = accuracy_score(y_test, pred_tab_only)
    
    # Stacked prediction
    X_meta_test = np.hstack([test_text_probs, test_tab_probs])
    pred_stacked = best_meta_model.predict(X_meta_test)
    acc_stacked = accuracy_score(y_test, pred_stacked)
    
    print(f"\nText only:     {acc_text_only:.4f}")
    print(f"Tabular only:  {acc_tab_only:.4f}")
    print(f"Stacked:       {acc_stacked:.4f}")
    
    print("\nConfusion Matrix (Stacked):")
    print(confusion_matrix(y_test, pred_stacked))
    
    print("\nClassification Report (Stacked):")
    print(classification_report(y_test, pred_stacked, target_names=classes))
    
    # =============================
    # SAVE RESULTS
    # =============================
    
    results = {
        "text_cv_scores": [float(s) for s in text_cv_scores],
        "text_cv_mean": float(np.mean(text_cv_scores)),
        "text_cv_std": float(np.std(text_cv_scores)),
        
        "tab_cv_scores": [float(s) for s in tab_cv_scores],
        "tab_cv_mean": float(np.mean(tab_cv_scores)),
        "tab_cv_std": float(np.std(tab_cv_scores)),
        
        "test_accuracy_text_only": float(acc_text_only),
        "test_accuracy_tab_only": float(acc_tab_only),
        "test_accuracy_stacked": float(acc_stacked),
        
        "best_meta_model": best_meta_name,
        "best_text_params": best_text_params,
        "best_tab_params": best_tab_params
    }
    
    with open(os.path.join(RESULTS_DIR, "final_results.json"), "w") as f:
        json.dump(results, f, indent=2)
    
    # Save models
    try:
        import joblib
        joblib.dump(grid_text.best_estimator_, 
                   os.path.join(RESULTS_DIR, "text_model.joblib"))
        joblib.dump(grid_tab.best_estimator_,
                   os.path.join(RESULTS_DIR, "tab_model.joblib"))
        joblib.dump(best_meta_model,
                   os.path.join(RESULTS_DIR, "meta_model.joblib"))
        print(f"\nModels saved to {RESULTS_DIR}")
    except Exception as e:
        print(f"Error saving models: {e}")
    
    return results

# -------------------------
# MAIN
# -------------------------

if __name__ == "__main__":
    df = pd.read_csv(TRAINING_DATA)
    df.columns = [
        'id',
        'best_tasks_free',
        'acad_tasks_rating',
        'best_tasks_select',
        'subopt_freq_rating',
        'subopt_tasks_select',
        'subopt_tasks_free',
        'evidence_freq_rating',
        'verify_freq_rating',
        'verify_method_free',
        'target'
    ]
    
    text_cols = [
        'best_tasks_free',
        'subopt_tasks_free',
        'verify_method_free'
    ]
    
    ord_cols = [
        'acad_tasks_rating',
        'subopt_freq_rating',
        'evidence_freq_rating',
        'verify_freq_rating'
    ]
    
    cat_cols = [
        'best_tasks_select',
        'subopt_tasks_select'
    ]
    
    results = run_improved_pipeline(df, text_cols, ord_cols, cat_cols)
    
    print(f"\n{'='*60}")
    print("PIPELINE COMPLETE")
    print(f"{'='*60}")
    print(f"Final stacked accuracy: {results['test_accuracy_stacked']:.4f}")


Train: 576 samples, Test: 249 samples
Classes: ['ChatGPT', 'Claude', 'Gemini']
Train distribution:
target
ChatGPT    192
Claude     192
Gemini     192
Name: count, dtype: int64

PHASE 1: TUNING BASE MODELS
Tuning TEXT model with expanded grid...
Fitting 5 folds for each of 720 candidates, totalling 3600 fits
[CV] END clf__C=0.05, clf__class_weight=None, clf__penalty=l2, clf__solver=lbfgs, tfidf__max_df=0.95, tfidf__max_features=3000, tfidf__min_df=1, tfidf__ngram_range=(1, 1), tfidf__sublinear_tf=True; total time=   0.1s
[CV] END clf__C=0.05, clf__class_weight=None, clf__penalty=l2, clf__solver=lbfgs, tfidf__max_df=0.95, tfidf__max_features=3000, tfidf__min_df=1, tfidf__ngram_range=(1, 1), tfidf__sublinear_tf=True; total time=   0.1s


/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/l

[CV] END clf__C=0.05, clf__class_weight=None, clf__penalty=l2, clf__solver=lbfgs, tfidf__max_df=0.95, tfidf__max_features=3000, tfidf__min_df=1, tfidf__ngram_range=(1, 1), tfidf__sublinear_tf=True; total time=   0.1s
[CV] END clf__C=0.05, clf__class_weight=None, clf__penalty=l2, clf__solver=lbfgs, tfidf__max_df=0.95, tfidf__max_features=3000, tfidf__min_df=1, tfidf__ngram_range=(1, 1), tfidf__sublinear_tf=True; total time=   0.1s
[CV] END clf__C=0.05, clf__class_weight=None, clf__penalty=l2, clf__solver=lbfgs, tfidf__max_df=0.95, tfidf__max_features=3000, tfidf__min_df=1, tfidf__ngram_range=(1, 1), tfidf__sublinear_tf=True; total time=   0.1s
[CV] END clf__C=0.05, clf__class_weight=None, clf__penalty=l2, clf__solver=lbfgs, tfidf__max_df=0.95, tfidf__max_features=3000, tfidf__min_df=1, tfidf__ngram_range=(1, 1), tfidf__sublinear_tf=False; total time=   0.1s


/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


[CV] END clf__C=0.05, clf__class_weight=None, clf__penalty=l2, clf__solver=lbfgs, tfidf__max_df=0.95, tfidf__max_features=3000, tfidf__min_df=1, tfidf__ngram_range=(1, 2), tfidf__sublinear_tf=True; total time=   0.1s
[CV] END clf__C=0.05, clf__class_weight=None, clf__penalty=l2, clf__solver=lbfgs, tfidf__max_df=0.95, tfidf__max_features=3000, tfidf__min_df=1, tfidf__ngram_range=(1, 2), tfidf__sublinear_tf=True; total time=   0.1s
[CV] END clf__C=0.05, clf__class_weight=None, clf__penalty=l2, clf__solver=lbfgs, tfidf__max_df=0.95, tfidf__max_features=3000, tfidf__min_df=1, tfidf__ngram_range=(1, 2), tfidf__sublinear_tf=False; total time=   0.1s


...

Text CV:    0.5907 ± 0.0352
Tabular CV: 0.6644 ± 0.0621

PHASE 3: STACKING ENSEMBLE


/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/l

Meta-model 'logistic': 0.6574 ± 0.0628
Meta-model 'logistic_balanced': 0.6574 ± 0.0628
Meta-model 'gradient_boost': 0.6614 ± 0.0314

Best meta-model: gradient_boost

PHASE 4: FINAL TEST EVALUATION


/Users/vic/miniconda3/envs/sklearn/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



Text only:     0.6185
Tabular only:  0.6787
Stacked:       0.6506

Confusion Matrix (Stacked):
[[69  7  7]
 [ 9 44 30]
 [ 9 25 49]]

Classification Report (Stacked):
              precision    recall  f1-score   support

     ChatGPT       0.79      0.83      0.81        83
      Claude       0.58      0.53      0.55        83
      Gemini       0.57      0.59      0.58        83

    accuracy                           0.65       249
   macro avg       0.65      0.65      0.65       249
weighted avg       0.65      0.65      0.65       249

Error saving models: Can't pickle <function preprocess_ord.<locals>.impute_and_scale at 0x16b7f40d0>: it's not found as __main__.preprocess_ord.<locals>.impute_and_scale

PIPELINE COMPLETE
Final stacked accuracy: 0.6506
